# 02 — Knowledge Extraction
Parses the collected corpus into structured, queryable knowledge:
- Action metadata from Swift source (verbs, roles, prepositions)
- Q&A seeds from Proposals (headings + code blocks)
- Example pairs (code → expected output) from Examples/

**Input:**  `../data/01_corpus/manifest.json`
**Output:** `../data/02_knowledge/knowledge.json`

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json
import re
from pathlib import Path
from collections import defaultdict

DATA_OUT = GLOBAL_OUT_DIR / '../data/02_knowledge'
DATA_OUT.mkdir(parents=True, exist_ok=True)

with open(DATA_IN / 'manifest.json') as f:
    corpus = json.load(f)

print(f'Loaded manifest: {len(corpus["aro_files"])} .aro files, '
      f'{len(corpus["proposals"])} proposals, {len(corpus["swift_actions"])} Swift files')

## Extract action metadata from Swift source

In [ ]:
def extract_action_metadata(swift_path):
    content = Path(swift_path).read_text()
    name = Path(swift_path).stem

    verbs_m  = re.search(r'static let verbs[^=]*=\s*\[([^\]]+)\]', content, re.DOTALL)
    role_m   = re.search(r'static let role[^=]*=\s*\.(\w+)', content)
    prep_m   = re.search(r'static let validPrepositions[^=]*=\s*\[([^\]]+)\]', content, re.DOTALL)
    desc_m   = re.search(r'//\s*(.+?)\n\s*(?:public\s+)?struct\s+\w+Action', content)

    verbs        = re.findall(r'"([^"]+)"', verbs_m.group(1)) if verbs_m else []
    role         = role_m.group(1) if role_m else 'unknown'
    prepositions = re.findall(r'\.(\w+)', prep_m.group(1)) if prep_m else []
    description  = desc_m.group(1).strip() if desc_m else ''

    return {
        'name': name,
        'verbs': verbs,
        'role': role,
        'prepositions': prepositions,
        'description': description,
    }

actions = [extract_action_metadata(f) for f in corpus['swift_actions']]
actions = [a for a in actions if a['verbs']]

print(f'Extracted {len(actions)} action definitions')
for a in actions[:5]:
    print(f'  {a["name"]}: verbs={a["verbs"]}, role={a["role"]}')

## Extract Q&A seeds from Proposals

In [ ]:
def extract_from_proposal(md_path):
    content = Path(md_path).read_text()
    name = Path(md_path).stem

    # All ARO code blocks in this proposal
    aro_blocks = re.findall(r'```aro\n(.*?)```', content, re.DOTALL)

    # Split into sections by heading
    section_pattern = re.compile(r'^(#{1,3} .+)', re.MULTILINE)
    parts = section_pattern.split(content)

    qa_seeds = []
    i = 1
    while i < len(parts):
        heading_raw = parts[i]
        body = parts[i + 1] if i + 1 < len(parts) else ''
        heading = re.sub(r'^#+\s*', '', heading_raw).strip()
        section_code = re.findall(r'```aro\n(.*?)```', body, re.DOTALL)

        if len(body.strip()) > 80:
            qa_seeds.append({
                'heading':      heading,
                'body':         body.strip()[:2000],
                'code_examples': [c.strip() for c in section_code],
                'proposal':     name,
            })
        i += 2

    return {
        'name':           name,
        'aro_code_blocks': [b.strip() for b in aro_blocks],
        'qa_seeds':        qa_seeds,
    }

proposals_knowledge = [extract_from_proposal(f) for f in corpus['proposals']]

total_seeds  = sum(len(p['qa_seeds']) for p in proposals_knowledge)
total_blocks = sum(len(p['aro_code_blocks']) for p in proposals_knowledge)
print(f'Proposals: {len(proposals_knowledge)} files, {total_seeds} Q&A seeds, {total_blocks} code blocks')

## Load examples with expected output

In [ ]:
def load_example(example_dir):
    p = Path(example_dir)
    aro_files = {}
    for f in sorted(p.rglob('*.aro')):
        rel = str(f.relative_to(p))
        aro_files[rel] = f.read_text()

    def read_opt(filename):
        fp = p / filename
        return fp.read_text() if fp.exists() else None

    return {
        'name':      p.name,
        'dir':       str(p),
        'aro_files': aro_files,
        'openapi':   read_opt('openapi.yaml'),
        'expected':  read_opt('expected.txt'),
        'readme':    read_opt('README.md'),
        'test_hint': read_opt('test.hint'),
    }

# Discover example root directories
# An example root is a dir that directly contains .aro files or has expected.txt
example_dirs = set()
for f_str in corpus['aro_files']:
    p = Path(f_str)
    # Walk up to find the highest directory that still has .aro files
    # but whose parent is one of the known collection roots
    candidate = p.parent
    while True:
        parent = candidate.parent
        if parent.name in ('Examples', 'ARO-Application', 'sources'):
            break
        if not list(parent.glob('*.aro')) and not (parent / 'expected.txt').exists():
            break
        candidate = parent
    example_dirs.add(str(candidate))

examples = [load_example(d) for d in sorted(example_dirs)]
examples = [e for e in examples if e['aro_files']]

print(f'Loaded {len(examples)} examples')
print(f'  With expected output: {sum(1 for e in examples if e["expected"])}')
print(f'  With openapi.yaml:    {sum(1 for e in examples if e["openapi"])}')

## Save knowledge base

In [ ]:
knowledge = {
    'actions':            actions,
    'proposals':          proposals_knowledge,
    'examples':           examples,
    'aro_syntax':         corpus['aro_syntax'],
    'aro_actions':        corpus['aro_actions'],
    'extra_docs':         corpus.get('extra_docs', {}),
}

out_path = DATA_OUT / 'knowledge.json'
with open(out_path, 'w') as f:
    json.dump(knowledge, f, indent=2)

print(f'Saved: {out_path}')
print(f'  {len(actions)} actions')
print(f'  {total_seeds} Q&A seeds from proposals')
print(f'  {len(examples)} examples')